# SFT、LoRA 与 DPO：损失、梯度和数据边界

标准库原创机制实验，无模型、服务、GPU 或训练框架。先预测结果，再运行；断言失败也是需要保留的证据。最后生成可导入本站笔记的 JSON。

## 1. 监督 mask 改变目标，零监督显式报错

In [ ]:
import math
def masked_nll(log_probs,mask):
    if len(log_probs)!=len(mask) or not mask or any(type(m) is not int or m not in (0,1) for m in mask):
        raise ValueError('binary mask with matching length required')
    if not all(math.isfinite(v) and v<=0 for v in log_probs):raise ValueError('finite log probabilities required')
    if not sum(mask):raise ValueError('no supervised targets')
    return -sum(p*m for p,m in zip(log_probs,mask))/sum(mask)
def rejects(fn):
    try:fn()
    except ValueError:return True
    raise AssertionError('negative case was accepted')
log_probs=[-2,-.2,-.4,-3]
assert math.isclose(masked_nll(log_probs,[0,1,1,0]),.3)
assert math.isclose(masked_nll(log_probs,[1,1,1,1]),1.4)
assert rejects(lambda:masked_nll(log_probs,[0,0,0,0]))
assert rejects(lambda:masked_nll(log_probs,[0,1,1]))
print('assistant targets:',masked_nll(log_probs,[0,1,1,0]))

## 2. LoRA 前向与解析梯度；有限差分独立核对

In [ ]:
# One output, two inputs, rank=1, alpha/r=1.
W=[.2,-.1];A=[.3,-.2];B=0.0
samples=[([1.0,0.0],1.0),([0.0,1.0],-.5)]
def loss_grad(a,b):
    loss=0.;ga=[0.,0.];gb=0.
    for x,y in samples:
        ax=sum(ai*xi for ai,xi in zip(a,x))
        prediction=sum(w*xi for w,xi in zip(W,x))+b*ax
        error=prediction-y
        loss+=.5*error*error/len(samples)
        gb+=error*ax/len(samples)
        for j in range(len(a)):ga[j]+=error*b*x[j]/len(samples)
    return loss,ga,gb
initial,ga,gb=loss_grad(A,B)
assert ga==[0.,0.] and gb!=0.
a=[.31,-.21];b=.4;eps=1e-6
_,grad_a,grad_b=loss_grad(a,b)
for j in range(2):
    plus=a[:];minus=a[:];plus[j]+=eps;minus[j]-=eps
    numerical=(loss_grad(plus,b)[0]-loss_grad(minus,b)[0])/(2*eps)
    assert math.isclose(numerical,grad_a[j],rel_tol=1e-7,abs_tol=1e-9)
numerical_b=(loss_grad(a,b+eps)[0]-loss_grad(a,b-eps)[0])/(2*eps)
assert math.isclose(numerical_b,grad_b,rel_tol=1e-7,abs_tol=1e-9)
original_w=W[:]
for step in range(200):
    loss,ga,gb=loss_grad(A,B)
    A=[v-.1*g for v,g in zip(A,ga)];B-=.1*gb
assert loss_grad(A,B)[0]<initial/10
assert W==original_w
print('initial/final loss:',initial,loss_grad(A,B)[0],'; only tiny regression, no LM training')

## 3. LoRA 参数数量只对一个目标矩阵成立

In [ ]:
rank=8;d_in=d_out=4096
adapter_parameters=rank*(d_in+d_out)
full_parameters=d_in*d_out
assert adapter_parameters==65536 and full_parameters==16777216
assert full_parameters/adapter_parameters==256
print('trainable parameter ratio for one matrix:',full_parameters/adapter_parameters)
print('not a total-model VRAM ratio; activations and base weights remain')

## 4. DPO：reference 差、标签方向与数值稳定

In [ ]:
def dpo_loss(policy_chosen,policy_rejected,ref_chosen,ref_rejected,beta=.1):
    values=[policy_chosen,policy_rejected,ref_chosen,ref_rejected,beta]
    if not all(math.isfinite(v) for v in values) or beta<=0:raise ValueError('finite inputs and positive beta required')
    z=beta*((policy_chosen-policy_rejected)-(ref_chosen-ref_rejected))
    # softplus(-z), stable for very large positive/negative margins
    return max(-z,0)+math.log1p(math.exp(-abs(z)))
equal=dpo_loss(-2,-4,-2,-4)
better=dpo_loss(-2,-4,-3,-4)
swapped=dpo_loss(-4,-2,-4,-3)
assert math.isclose(equal,math.log(2))
assert math.isclose(better,.6443966600735709)
assert math.isclose(swapped,.7443966600735709)
assert math.isfinite(dpo_loss(-10000,-1,-1,-1))
assert rejects(lambda:dpo_loss(-2,-4,-3,-4,float('nan')))
eps=1e-5
numerical=(dpo_loss(-2+eps,-4,-3,-4)-dpo_loss(-2-eps,-4,-3,-4))/(2*eps)
analytical=-.1/(1+math.exp(.1))
assert math.isclose(numerical,analytical,rel_tol=1e-7)
print('equal/better/swapped loss:',equal,better,swapped)

## 5. 按来源划分：相邻 chunk 不可横跨训练和测试

In [ ]:
def validate_split(train,dev,test):
    groups=[set(s) for s in (train,dev,test)]
    if any(groups[i]&groups[j] for i in range(3) for j in range(i+1,3)):
        raise ValueError('source groups leaked across splits')
    return True
assert validate_split(['doc-a','doc-b'],['doc-c'],['doc-d'])
assert rejects(lambda:validate_split(['doc-a'],['doc-b'],['doc-a']))
print('group split accepted; repeated source rejected')

## 修改与记录

改变一个输入，先记录预测，再重新执行有关格。不要直接将检查值改成 True。导入只校验格式，不证明真实模型能力。

In [ ]:
import json
from pathlib import Path
checks = {'loss_mask': True, 'lora_gradient': True, 'frozen_base': True, 'dpo_reference': True, 'group_split': True, 'negative_inputs': True}
report = {'experiment': 'lesson-finetuning-foundations', 'scope': 'stdlib fixtures; no model/service',
          'checks': checks, 'my_explanation': ''}
Path('mechanism_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2))
print(json.dumps(report, ensure_ascii=False, indent=2))